# Distance-along-shape exploration

Projects Culver City vehicle positions onto their route shapes, smooths the per-trip trajectories, projects nearby fixed points (signals and stops), estimates per-signal intersection delay, and renders diagnostic plots and a folium map for a single trip.

In [ ]:
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import LineString

from src.constants import (
    CA_NAD83_Albers,
    CULVER_CITY_FEED_KEY,
    MAX_SNAP_DISTANCE_M,
    SERVICE_DATE,
    SHAPE_KEY_TO_SHAPE_ID_MAP,
)
from src._data_loaders import (
    get_matched_vehicle_positions,
    get_selected_shapes,
    get_smoothed_vehicle_positions,
    get_traffic_signals,
)
from src.match_shapes_vp import project_points_on_shape
from src.delay_calculation import estimate_intersection_time_single_trip
from src.plot_trip import plot_distance_over_time, plot_speed_over_time


shape_ids = list(SHAPE_KEY_TO_SHAPE_ID_MAP.values())

In [ ]:
# for caltrans network only

import os

os.environ["REQUESTS_CA_BUNDLE"] = r"C:\Users\s163107\Documents\CTROOTCA01.cer"

## Configuration

Select the trip to analyze by `TRIP_KEY` on a `SERVICE_DATE`.

In [ ]:
# Select the trip to analyze: a TRIP_KEY on a SERVICE_DATE.
SERVICE_DATE = "2026-02-04"
TRIP_KEY = "986.0"  # e.g. "1026.0"; None plots the first available trip for the date

## Data loading

In [ ]:
vp = get_matched_vehicle_positions(SERVICE_DATE)

In [ ]:
vp[["ROUTE_ID", "TRIP_KEY"]].drop_duplicates(keep="first", ignore_index=True)

In [ ]:
vp.DWELL_TIME.loc[vp.DWELL_TIME.notna()]

In [ ]:
shapes = get_selected_shapes(SERVICE_DATE, CULVER_CITY_FEED_KEY, shape_ids)

In [ ]:
signals = get_traffic_signals()

In [ ]:
stops = gpd.read_file("data/stops_shp-1-05.geojson").to_crs(CA_NAD83_Albers)

## Trip smoothing

In [ ]:
# Per-trip smoothed distance-along-shape and speed are precomputed across all
# trips/dates by 0_map_match_and_smooth.py.
smoothed = get_smoothed_vehicle_positions(SERVICE_DATE)
vp_smoothed = smoothed
vp_speeds = smoothed

In [ ]:
smoothed

In [ ]:
shape_geom = shapes.set_index("shape_id")["geometry"]
vp_shape_ids = vp["ROUTE_ID"].str.strip().map(SHAPE_KEY_TO_SHAPE_ID_MAP)
snap_distances = vp.groupby(vp_shape_ids, group_keys=False).apply(
    lambda g: g.geometry.map(lambda pt: pt.distance(shape_geom[g.name].interpolate(shape_geom[g.name].project(pt))))
)
print("Distance from vehicle position to projected shape (meters) — all points:")
print(snap_distances.describe())
print()
print(f"Distance from vehicle position to projected shape (meters) — within {MAX_SNAP_DISTANCE_M}m threshold:")
print(snap_distances[vp["distance_along_shape"].notna()].describe())

## Single-trip selection

In [ ]:
# Select the trip set in the configuration cell (TRIP_KEY on SERVICE_DATE).
available_trip_keys = sorted(vp_smoothed["TRIP_KEY"].unique())
if TRIP_KEY is None:
    trip_key = available_trip_keys[0]
elif TRIP_KEY not in available_trip_keys:
    raise ValueError(
        f"TRIP_KEY {TRIP_KEY!r} not found for {SERVICE_DATE}. "
        f"{len(available_trip_keys)} available, e.g. {available_trip_keys[:10]}"
    )
else:
    trip_key = TRIP_KEY
print(f"Plotting trip {trip_key} on {SERVICE_DATE}")

trip = vp[vp["TRIP_KEY"] == trip_key].sort_values("event_time_datetime")
trip_smoothed = vp_smoothed[vp_smoothed["TRIP_KEY"] == trip_key].sort_values("event_time_datetime")
trip_speeds = vp_speeds[vp_speeds["TRIP_KEY"] == trip_key].sort_values("event_time_datetime")

route_id = trip["ROUTE_ID"].iloc[0].strip()
trip_shape = shape_geom[[SHAPE_KEY_TO_SHAPE_ID_MAP[route_id]]]

## Ping data quality

Time and physical distance between consecutive raw GPS pings for the selected trip (before smoothing).

In [ ]:
# Selected trip is already sorted by event_time_datetime.
time_between_pings_s = trip["event_time_datetime"].diff().dt.total_seconds()
distance_between_pings_m = trip.geometry.distance(trip.geometry.shift())

ping_quality = pd.DataFrame(
    {
        "time_between_pings_s": time_between_pings_s,
        "distance_between_pings_m": distance_between_pings_m,
    }
).describe(percentiles=[0.5, 0.8, 0.9, 0.95, 0.99])
ping_quality

## Fixed-point projection

Project signals and stops onto the trip's shape geometry, then estimate per-signal delay for this trip.

In [ ]:
STOP_PLOT_SNAP_DISTANCE_M = 5

signal_distances = project_points_on_shape(signals, trip_shape, MAX_SNAP_DISTANCE_M)
nearside_stop_distances = project_points_on_shape(
    stops[stops["nearside"] == True], trip_shape, STOP_PLOT_SNAP_DISTANCE_M
)
farside_stop_distances = project_points_on_shape(
    stops[stops["nearside"] != True], trip_shape, STOP_PLOT_SNAP_DISTANCE_M
)

In [ ]:
NEARSIDE_STOP_DISTANCE_M = 30
SIGNAL_EFFECT_DISTANCE_M = 100

trip_stops_projected = project_points_on_shape(stops, trip_shape, MAX_SNAP_DISTANCE_M)
trip_stops_nearside = stops.loc[trip_stops_projected.index, "nearside"]
signal_delays_s = estimate_intersection_time_single_trip(
    smoothed_trajectory=trip_smoothed.set_index("event_time_datetime")["distance_along_shape_smoothed"],
    smoothed_speeds_mps=trip_speeds.set_index("event_time_datetime")["speed_m_per_s"],
    stops_projected=trip_stops_projected,
    stops_nearside=trip_stops_nearside,
    signals_projected=signal_distances,
    nearside_stop_distance=NEARSIDE_STOP_DISTANCE_M,
    signal_effect_distance=SIGNAL_EFFECT_DISTANCE_M,
)

## Plots

In [ ]:
M_PER_MILE = 1609.344
SECONDS_PER_HOUR = 3600
HIGHLIGHT_DELAY_THRESHOLD_S = 1.0

### Distance along shape over time

In [ ]:
# from src.smooth_trajectory import smooth_distances_per_trip

# trip_smoothed = smooth_distances_per_trip(trip, trip["distance_along_shape"], 1)
# trip_smoothed_merged = trip.merge(trip_smoothed, on=["TRIP_KEY", "event_time_datetime"], how="inner")
# trip_smoothed_merged["error"] = abs(
#     trip_smoothed_merged["distance_along_shape"] - trip_smoothed_merged["distance_along_shape_smoothed"]
# )
# trip_smoothed_merged
# plt.plot(trip_smoothed_merged["distance_along_shape"], trip_smoothed_merged["error"])

In [ ]:
from src.smooth_trajectory import smooth_distances_per_trip


fig_dist, ax_dist = plot_distance_over_time(
    trips=[trip],
    signal_distances=signal_distances,
    nearside_stop_distances=nearside_stop_distances,
    farside_stop_distances=farside_stop_distances,
    signal_delays_s=signal_delays_s,
    trips_smoothed=[smooth_distances_per_trip(trip, trip["distance_along_shape"], 1)],
    highlight_delay_threshold_s=HIGHLIGHT_DELAY_THRESHOLD_S,
)
fig_dist.savefig("test_distance_along_shape_smoothed.png", dpi=150)
plt.show()

In [ ]:
# investigating pattern of jumps
jump_area = trip.loc[
    (trip["distance_along_shape"] > 8360) & (trip["distance_along_shape"] < 9334) & trip["distance_along_shape"].notna()
]
(
    trip["TRIP_KEY"].iloc[0],
    SERVICE_DATE,
    min(jump_area["EVENT_TIME_UTC"]),
    max(jump_area["EVENT_TIME_UTC"]),
    len(jump_area),
)

In [ ]:
jump_area["event_time_datetime"].dt.date

### Speed over time

In [ ]:
# Project each signal/stop distance to the time the bus crossed that distance
smoothed_trip_distances = trip_smoothed.set_index("event_time_datetime")["distance_along_shape_smoothed"]

signal_crossing_times = distances_to_crossing_times(signal_distances, smoothed_trip_distances)
nearside_stop_crossing_times = distances_to_crossing_times(nearside_stop_distances, smoothed_trip_distances)
farside_stop_crossing_times = distances_to_crossing_times(farside_stop_distances, smoothed_trip_distances)

In [ ]:
signal_distances

In [ ]:
speed_mph = trip_speeds["speed_m_per_s"] * SECONDS_PER_HOUR / M_PER_MILE

trip_start = trip_speeds["event_time_datetime"].min()
signal_crossing_elapsed_s = (signal_crossing_times - trip_start).dt.total_seconds()
nearside_stop_crossing_elapsed_s = (nearside_stop_crossing_times - trip_start).dt.total_seconds()
farside_stop_crossing_elapsed_s = (farside_stop_crossing_times - trip_start).dt.total_seconds()

fig_speed, ax_speed = plot_speed_over_time(
    trips_speeds=[trip_speeds],
    signal_crossing_times_elapsed_s=signal_crossing_elapsed_s,
    signal_delays_s=signal_delays_s,
    nearside_stop_crossing_times_elapsed_s=nearside_stop_crossing_elapsed_s,
    farside_stop_crossing_times_elapsed_s=farside_stop_crossing_elapsed_s,
    highlight_delay_threshold_s=HIGHLIGHT_DELAY_THRESHOLD_S,
)
fig_speed.savefig("test_speed_over_time_smoothed.png", dpi=150)
plt.show()

### Folium map: signals, stops, and speed-colored trip path

In [ ]:
trip_shape_linestring = trip_shape.iloc[0]

trip_path_points_geodesic = gpd.GeoSeries(
    [trip_shape_linestring.interpolate(d) for d in trip_smoothed["distance_along_shape_smoothed"]],
    crs=CA_NAD83_Albers,
).to_crs("EPSG:4326")

speed_values_mph = speed_mph.to_numpy()
segment_speeds_mph = (speed_values_mph[:-1] + speed_values_mph[1:]) / 2
segment_geometries = [
    LineString([trip_path_points_geodesic.iloc[i], trip_path_points_geodesic.iloc[i + 1]])
    for i in range(len(trip_path_points_geodesic) - 1)
]
trip_segments_geodesic = gpd.GeoDataFrame(
    {
        "speed_mph": segment_speeds_mph,
        "speed_label": [f"{s:.1f} mph" for s in segment_speeds_mph],
    },
    geometry=segment_geometries,
    crs="EPSG:4326",
)

speed_colormap = plt.get_cmap("RdYlBu_r")  # blue=slow, red=fast
speed_color_norm = mcolors.Normalize(vmin=speed_values_mph.min(), vmax=speed_values_mph.max())

map_center_lat = trip_path_points_geodesic.geometry.y.mean()
map_center_lon = trip_path_points_geodesic.geometry.x.mean()
folium_map = folium.Map(location=[map_center_lat, map_center_lon], zoom_start=14, tiles="CartoDB positron")

folium.GeoJson(
    trip_segments_geodesic,
    style_function=lambda feature: {
        "color": mcolors.to_hex(speed_colormap(speed_color_norm(feature["properties"]["speed_mph"]))),
        "weight": 5,
        "opacity": 0.85,
    },
    tooltip=folium.GeoJsonTooltip(fields=["speed_label"], aliases=["speed"]),
).add_to(folium_map)

signals_near_shape_geodesic = (
    signals.loc[signal_distances.index]
    .to_crs("EPSG:4326")
    .assign(delay_text=signal_delays_s.apply(lambda s: f"{s:.1f} s"))
)
is_high_delay_signal = signal_delays_s > HIGHLIGHT_DELAY_THRESHOLD_S
low_delay_signals_geodesic = signals_near_shape_geodesic.loc[~is_high_delay_signal]
high_delay_signals_geodesic = signals_near_shape_geodesic.loc[is_high_delay_signal]
if not low_delay_signals_geodesic.empty:
    folium.GeoJson(
        low_delay_signals_geodesic,
        marker=folium.CircleMarker(radius=3, color="gray", fill=True, fill_opacity=0.85),
        tooltip=folium.GeoJsonTooltip(fields=["delay_text"], aliases=["signal delay"]),
    ).add_to(folium_map)
if not high_delay_signals_geodesic.empty:
    folium.GeoJson(
        high_delay_signals_geodesic,
        marker=folium.CircleMarker(radius=5, color="black", weight=2, fill=True, fill_color="gray", fill_opacity=0.85),
        tooltip=folium.GeoJsonTooltip(fields=["delay_text"], aliases=["signal delay"]),
    ).add_to(folium_map)

nearside_stops_geodesic = stops.loc[nearside_stop_distances.index].to_crs("EPSG:4326")
folium.GeoJson(
    nearside_stops_geodesic,
    marker=folium.CircleMarker(radius=4, color="blue", fill=True, fill_opacity=0.85),
    tooltip=folium.GeoJsonTooltip(fields=["stop_name"], aliases=["near-side stop"]),
).add_to(folium_map)

farside_stops_geodesic = stops.loc[farside_stop_distances.index].to_crs("EPSG:4326")
folium.GeoJson(
    farside_stops_geodesic,
    marker=folium.CircleMarker(radius=4, color="pink", fill=True, fill_opacity=0.85),
    tooltip=folium.GeoJsonTooltip(fields=["stop_name"], aliases=["far-side stop"]),
).add_to(folium_map)

folium_map.save("test_trip_speeds_map.html")
folium_map